# Projet de Science des Données : Classification d'Assurance Automobile
**Auteurs** : Étudiant (accompagné par Gemini CLI)  
**Sujet** : Prédiction de réclamations d'assurance (Classification supervisée)

---

## 1. Introduction et Objectif
L'objectif de ce projet est de construire un modèle capable de prédire si un client d'assurance automobile fera une demande d'indemnisation (**Outcome = 1**) ou non (**Outcome = 0**). 

Nous suivons les étapes classiques d'un projet de Data Science : importation, examen, préparation, analyse de corrélations, modélisation et évaluation.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import warnings
from pandas.plotting import scatter_matrix
import os

# Imports scikit-learn
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.linear_model import LogisticRegression, Perceptron
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (accuracy_score, confusion_matrix, precision_score, 
                            recall_score, f1_score, classification_report)

warnings.filterwarnings('ignore')
%matplotlib inline
sns.set_theme(style="whitegrid")

# Création des dossiers de sortie si nécessaire
os.makedirs('models', exist_ok=True)
os.makedirs('output/plots', exist_ok=True)

## 2. Importation et Examen des données (Étapes 3 & 4 du sujet)

Nous chargeons les données et vérifions leur structure initiale.

In [ ]:
# Chargement (Données situées dans le dossier /data)
df = pd.read_csv("data/car_insurance.csv")
print(f"Taille du jeu de données : {df.shape[0]} lignes, {df.shape[1]} variables")
df.head()

In [ ]:
# Informations sur les types et données manquantes
df.info()

# Vérification explicite des données manquantes (Étape 4 du sujet)
print("\nNombre de valeurs manquantes par variable :")
print(df.isna().sum())

In [ ]:
# Statistiques descriptives pour détecter d'éventuelles valeurs aberrantes
df.describe()

### Examen visuel (Histogrammes)
Nous traçons les histogrammes pour comprendre la distribution de chaque variable numérique.

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns
df[numeric_cols].hist(figsize=(15, 12), bins=30, edgecolor='black')
plt.tight_layout()
plt.savefig('output/plots/histogrammes_variables.png')
plt.show()

### Observations et Données Aberrantes (Étape 4 du sujet)

En examinant les histogrammes et les statistiques descriptives, nous observons :
1. **Distributions** : Certaines variables comme `credit_score` suivent une distribution proche de la normale. D'autres sont fortement décalées à gauche (ex: `speeding_violations`, `past_accidents`).
2. **Données Aberrantes** : La variable `speeding_violations` présente des valeurs extrêmement élevées (jusqu'à 41 056), ce qui est physiquement impossible et constitue des données aberrantes. 
3. **Données Manquantes** : Nous avons identifié des valeurs manquantes dans `credit_score` et `annual_mileage` (environ 10% du jeu de données chacune).

## 3. Préparation des données (Étape 5 du sujet)

Nous procédons au nettoyage : suppression des colonnes inutiles, traitement des valeurs manquantes, encodage et normalisation.

In [ ]:
df_prep = df.copy()

# 1. Identification des variables qualitatives (Étape 4)
qual_cols = df_prep.select_dtypes(include=['object']).columns.tolist()
print(f"Variables qualitatives identifiées : {qual_cols}")

# 2. Suppression des colonnes non pertinentes
df_prep = df_prep.drop(columns=['id', 'postal_code'])

# 3. Traitement des données aberrantes (Étape 5)
# On remplace les valeurs de speeding_violations > 50 par la médiane
median_speeding = df_prep['speeding_violations'].median()
df_prep.loc[df_prep['speeding_violations'] > 50, 'speeding_violations'] = median_speeding

# 4. Traitement des données manquantes (Étape 5 - Solution à privilégier : fillna)
# Remplacement par la médiane pour les variables numériques
df_prep['credit_score'] = df_prep['credit_score'].fillna(df_prep['credit_score'].median())
df_prep['annual_mileage'] = df_prep['annual_mileage'].fillna(df_prep['annual_mileage'].median())

# 5. Encodage des variables qualitatives (LabelEncoder)
label_encoders = {}
for col in df_prep.select_dtypes(include=['object']).columns:
    le = LabelEncoder()
    df_prep[col] = le.fit_transform(df_prep[col].astype(str))
    label_encoders[col] = le

# 6. Normalisation (StandardScaler)
X = df_prep.drop(columns=['outcome'])
y = df_prep['outcome']
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

print(f"Données après préparation : {df_prep.shape[0]} lignes")

## 4. Recherche de corrélations (Étape 6 du sujet)

Le coefficient de corrélation (souvent celui de Pearson) mesure la force et la direction de la relation linéaire entre deux variables. Il est compris entre -1 et 1 :
- **1** : Corrélation positive parfaite.
- **0** : Aucune relation linéaire.
- **-1** : Corrélation négative parfaite.

Cela nous aide à identifier les variables les plus influentes pour notre cible `outcome`.

In [ ]:
# Matrice de corrélation
plt.figure(figsize=(12, 8))
sns.heatmap(pd.concat([X_scaled, y.reset_index(drop=True)], axis=1).corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title("Matrice de Corrélation")
plt.savefig('output/plots/correlation_matrix.png')
plt.show()

**Analyse des corrélations** : 
Les variables les plus corrélées avec `outcome` sont `past_accidents` (0.52), `speeding_violations` (0.48) et `duis` (0.42). 

Visualisons ces variables prometteuses avec une matrice de dispersion.

In [ ]:
top_vars = ['past_accidents', 'speeding_violations', 'duis', 'annual_mileage']
scatter_matrix(df_prep[top_vars], figsize=(12, 8), alpha=0.3, diagonal='kde')
plt.show()

## 5. Division et Apprentissage (Étapes 7 & 8 du sujet)

Nous divisons les données : 70% pour l'apprentissage et 30% pour le test.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.3, random_state=42, stratify=y)

print(f"Proportion Apprentissage : {len(X_train)/len(X_scaled)*100:.0f}%")
print(f"Proportion Test : {len(X_test)/len(X_scaled)*100:.0f}%")

### Entraînement de la Régression Logistique

**Réponses aux questions théoriques** :
- **Hypothèse** : On suppose que le logarithme du rapport des vraisemblances (logit) est une combinaison linéaire des variables d'entrée.
- **Minimisation** : On utilise le maximum de vraisemblance, souvent optimisé par descente de gradient.
- **Apprentissage** : On calcule les coefficients (poids) $w$ et l'ordonnée à l'origine (intercept).

In [ ]:
log_reg = LogisticRegression(random_state=42)
log_reg.fit(X_train, y_train)
print("Modèle de régression logistique entraîné.")

## 6. Évaluation du modèle (Étape 9 du sujet)

Pour évaluer notre modèle, nous utilisons plusieurs métriques :
- **Accuracy** : Proportion de prédictions correctes sur le total.
- **Matrice de confusion** : Tableau montrant les Vrais Positifs, Vrais Négatifs, Faux Positifs et Faux Négatifs.
- **Precision** : Capacité à ne pas prédire un échantillon positif alors qu'il est négatif (Qualité).
- **Recall (Rappel)** : Capacité à trouver tous les échantillons positifs (Quantité).
- **F1-Score** : Moyenne harmonique de la précision et du rappel.

In [ ]:
y_pred = log_reg.predict(X_test)

print("Exemples de comparaisons (10 premiers échantillons) :")
for i in range(10):
    print(f"Échantillon {i}: Réel={y_test.iloc[i]}, Prédit={y_pred[i]}")

In [ ]:
# Métriques quantitatives
print("Accuracy :", accuracy_score(y_test, y_pred))
conf = confusion_matrix(y_test, y_pred)
print("\nMatrice de confusion :\n", conf)

# Visualisation de la matrice de confusion
plt.figure(figsize=(8, 6))
sns.heatmap(conf, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['No Claim', 'Made Claim'],
            yticklabels=['No Claim', 'Made Claim'])
plt.title('Matrice de Confusion - Régression Logistique')
plt.ylabel('Réel')
plt.xlabel('Prédit')
plt.savefig('output/plots/confusion_matrix.png')
plt.show()

print("\nRapport de classification :\n", classification_report(y_test, y_pred))

## 7. Amélioration et Comparaison (Étapes 10 & 11 du sujet)

Utilisons la validation croisée pour une évaluation plus robuste et comparons avec le Perceptron et les K-plus proches voisins (KNN).

In [ ]:
models = {
    'Régression Logistique': LogisticRegression(random_state=42),
    'Perceptron': Perceptron(random_state=42),
    'KNN (k=5)': KNeighborsClassifier(n_neighbors=5)
}

kfold = KFold(n_splits=5, shuffle=True, random_state=42)
results = {}

print("Scores de validation croisée (Accuracy) :")
for name, clf in models.items():
    scores = cross_val_score(clf, X_scaled, y, cv=kfold)
    results[name] = scores.mean()
    print(f"- {name} : {scores.mean():.4f} (+/- {scores.std():.4f})")

print("\nComparaison avec l'évaluation simple :")
print(f"Précision simple (LogReg) : {accuracy_score(y_test, y_pred):.4f}")
print(f"Précision CV (LogReg)     : {results['Régression Logistique']:.4f}")
print("La validation croisée donne une estimation plus fiable car elle utilise l'ensemble du jeu de données.")

**Conclusion de la comparaison** : La Régression Logistique offre généralement la meilleure précision et stabilité pour ce jeu de données.

## 8. Sauvegarde du modèle (Étape 12 du sujet)

Nous sauvegardons le meilleur modèle et les transformateurs pour le système en production.

In [ ]:
with open('models/best_model.pkl', 'wb') as f: pickle.dump(log_reg, f)
with open('models/scaler.pkl', 'wb') as f: pickle.dump(scaler, f)
with open('models/label_encoders.pkl', 'wb') as f: pickle.dump(label_encoders, f)
print("Objets sauvegardés avec Pickle dans le dossier /models.")